# CSE570 Unit II
## Part 3 — Imbalanced Data and Preprocessing Pipelines

In [ ]:
!pip -q install imbalanced-learn

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report
from imblearn.over_sampling import RandomOverSampler, SMOTE
from imblearn.under_sampling import RandomUnderSampler

In [ ]:
# This cell makes the notebook work both locally and in Google Colab.
from pathlib import Path
import pandas as pd

csv_name = "student_performance_eda.csv"
csv_path = Path(csv_name)

if not csv_path.exists():
    try:
        from google.colab import files
        print(f"Please upload {csv_name}")
        uploaded = files.upload()
        csv_path = Path(next(iter(uploaded.keys())))
    except ImportError:
        raise FileNotFoundError(
            f"{csv_name} was not found. Place it in the same folder as this notebook."
        )

df = pd.read_csv(csv_path)
print("Dataset loaded successfully.")
print("Shape:", df.shape)
df.head()

## Class distribution

In [ ]:
pd.DataFrame({
    "Count": df["Placed"].value_counts(),
    "Percentage": df["Placed"].value_counts(normalize=True).mul(100).round(2)
})

## Compare resampling methods

In [ ]:
X_resample = df.select_dtypes(include="number").drop(columns=["Student_ID"])
X_resample = X_resample.fillna(X_resample.median())
y_resample = df["Placed"].map({"No":0, "Yes":1})

samplers = {
    "Random Oversampling": RandomOverSampler(random_state=42),
    "Random Undersampling": RandomUnderSampler(random_state=42),
    "SMOTE": SMOTE(random_state=42)
}

for name, sampler in samplers.items():
    X_new, y_new = sampler.fit_resample(X_resample, y_resample)
    print(f"\n{name}")
    print(y_new.value_counts())

## Train-test split

In [ ]:
X = df.drop(columns=["Student_ID", "Placed"])
y = df["Placed"].map({"No":0, "Yes":1})

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)

## Identify numerical and categorical columns

In [ ]:
numerical_columns = X_train.select_dtypes(include="number").columns.tolist()
categorical_columns = X_train.select_dtypes(exclude="number").columns.tolist()
print("Numerical:", numerical_columns)
print("Categorical:", categorical_columns)

## Build preprocessing pipelines

In [ ]:
numerical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numerical_pipeline, numerical_columns),
    ("categorical", categorical_pipeline, categorical_columns)
])

## Complete ML pipeline

In [ ]:
model_pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("feature_selection", SelectKBest(score_func=f_classif, k=8)),
    ("model", LogisticRegression(class_weight="balanced", max_iter=1000))
])

model_pipeline.fit(X_train, y_train)
predictions = model_pipeline.predict(X_test)

print("Confusion Matrix")
print(confusion_matrix(y_test, predictions))

print("\nClassification Report")
print(classification_report(y_test, predictions, zero_division=0))

## Student Practice

In [ ]:
# Replace Logistic Regression with another classifier.
